<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

Which pages should a content reviewer open first? We studied 36,290 content items from 36 clients in the FlyRank ML Internship Warehouse (March 2026, 9.8 million daily performance rows), building a four-feature Random Forest that ranks pages by short-term decline risk. Under a client-grouped 70/30 validation split, the model achieved precision@50 = 0.760 against a test base rate of 0.477, outperforming both a hand-written rule baseline (0.520) and Logistic Regression (0.480). The output is a ranked review queue with reason codes — decision-support for a human reviewer, not an automated action system.

## 1. Question

**Research question:** Which published content items should a capacity-limited reviewer prioritise for refresh, ranked from observable search and engagement signals, and does a learned ranking beat a transparent rule baseline at precision@K?

**The decision:** out of tens of thousands of pages, which handful does a reviewer open first when they can check roughly 50 per cycle.

**Unit of analysis:** one content item, for one client, aggregated from daily search performance records over a 21-day feature window.

**Output:** a ranked review queue with a reason code on every page — `weak_position_high_demand` (refresh), `low_ctr_good_position` (optimize title/meta), or `thin_engagement` (review manually).

**Action:** a reviewer opens the flagged page, reads the reason code, and decides whether to refresh, expand, consolidate, or leave it. The model orders the queue; it never takes action.

**Cost of a wrong call:** a false positive wastes a reviewer's limited capacity on a page that was fine. A false negative lets a genuinely declining page bleed traffic unnoticed. Reviewer capacity is the binding constraint, so false positives are the more expensive error, and precision@K is the metric that matches.

**Why ML helps:** the hand-written "declining with demand" flag fires on 43.8% of pages in the starter slice — 13,152 of 30,000. Against a capacity of ~50, a flag that fires 13,152 times gives a reviewer no ordering. Every single-signal sort (position, CTR, impressions, word count) scores at or below the base rate at precision@50. If signal exists, it lives in combinations a threshold can't express.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Data

**Source:** FlyRank ML Internship Warehouse (`FlyRank/internship-warehouse` on Hugging Face), a gated, anonymized release of real production search and engagement data under the FlyRank Internship Data Use Terms.

**Scale:** 104 clients, ~520,000 content items, 78.8 million daily performance rows spanning 2025-01-27 to 2026-06-30.

**Tables used:**
- `fact_content_daily_performance` — daily search and engagement metrics per page per client (the primary table)
- `dim_content` — static page attributes (word count, content type, intent, dates)
- `dim_clients` — client metadata and data-availability dates

**Development month:** `month=2026-03` (9,841,378 rows). The final month (June 2026) was deliberately left sealed — it is the natural outcome window for any past→future label.

**What was excluded and why:**

| Excluded | Reason |
|---|---|
| `trend_direction`, `trend_pct` | Label-derived — they produce the starter proxy; using them as features would be circular |
| `optimization_eligible_date` | A product-rule output — learning from it means learning the rule the model is trying to beat |
| `is_deleted` | Reflects a decision already taken; deleted pages aren't review candidates |
| `provider_used`, `model_used` | Internal pipeline detail, not page performance |
| `last_optimized_date` | Signal check returned FALSE — 100% of non-null values fall *after* the decision moment (34–107 days later); forward-looking, not a refresh record |
| `client_hash_id`, `content_hash_id` | Salted pseudonyms used only for joining and splitting, never as features |
| AI-referral columns | 0.056% coverage in March — too sparse for features |

**Availability filter:** every query filters on `gsc_data_available IS TRUE`. Only 36.7% of March rows have GSC data; the rest represent clients without a Search Console connection, not pages without traffic. GA4 coverage is 4.2%.

**Public safety:** no client names, URLs, domains, or private queries appear anywhere in the repo or this report. All identifiers are salted pseudonymous hashes.

In [ ]:
!pip install -q duckdb

import duckdb, pandas as pd, numpy as np, json, os
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
HF_TOKEN = userdata.get("HF_TOKEN").strip()
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL       = "hf://datasets/FlyRank/internship-warehouse"
DEV_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
FEAT_END, OUT_START = "2026-03-21", "2026-03-22"

frame = con.sql(f"""
    WITH feat AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)            AS impressions_21d,
               SUM(gsc_clicks)                 AS clicks_21d,
               AVG(NULLIF(gsc_avg_position,0)) AS avg_position_21d,
               COUNT(*)                        AS days_observed
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date <= DATE '{FEAT_END}'
        GROUP BY 1,2
        HAVING SUM(gsc_clicks) > 0
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_out, COUNT(*) AS days_out
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date >= DATE '{OUT_START}'
        GROUP BY 1,2
    )
    SELECT f.*, o.clicks_out, o.days_out
    FROM feat f
    JOIN outcome o USING (client_hash_id, content_hash_id)
    WHERE o.days_out > 0 AND o.clicks_out > 0
""").df()

frame["ctr_21d"]      = frame.clicks_21d / frame.impressions_21d
frame["rate_before"]  = frame.clicks_21d / frame.days_observed
frame["rate_after"]   = frame.clicks_out / frame.days_out
frame["is_declining"] = (frame.rate_after < frame.rate_before).astype(int)

FEATURES = ["impressions_21d", "avg_position_21d", "ctr_21d", "days_observed"]
X, y, groups = frame[FEATURES], frame.is_declining, frame.client_hash_id

print(f"Frame: {len(frame):,} pages   Base rate: {y.mean():.1%}")
print(f"Clients: {groups.nunique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Frame: 36,290 pages   Base rate: 44.4%
Clients: 36


## 3. Methodology

**Assumptions.** A page's near-term click trajectory can be estimated from its recent search visibility and engagement patterns. The model's job is to order a queue, not to diagnose why a page is declining.

**Features (four, all from the 21-day feature window):**

| Feature | What it measures |
|---|---|
| `impressions_21d` | Search demand — how often the page appeared in results |
| `avg_position_21d` | Ranking quality — where it appeared (zeros excluded: `gsc_avg_position = 0` means no data) |
| `ctr_21d` | Click capture — what fraction of impressions converted to clicks |
| `days_observed` | Measurement density — how many of the 21 days had GSC data |

`clicks_21d` was originally a fifth feature (42.4% importance), but the ML-09 train-without audit found removing it *improved* precision@50 from 0.680 to 0.760. It was redundant with `impressions_21d` and `ctr_21d`, adding noise rather than signal.

**Label definition.** `is_declining = 1` when the daily click rate in the outcome window (days 22–31) falls below the daily click rate in the feature window (days 1–21). This is a proxy for short-term momentum loss. Both `clicks_21d > 0` and `clicks_out > 0` guards are applied — the label is undefined by construction outside this region (see Limitations).

**Baseline.** A hand-written rule scoring `impressions_21d`, `avg_position_21d`, and `ctr_21d` with fixed percentile-rank weights (0.3, 0.3, 0.4), three reason codes, and a deterministic tiebreak. Built in ML-07; precision@50 = 0.520 on the test split.

**Model.** Random Forest (200 trees, min_samples_leaf=20, seed=42). Chosen over Logistic Regression because the ML-07 top-10 review showed the baseline's weakness was an inability to capture feature interactions — pages with identical position and CTR levels could be declining or stable, and only combinations distinguish them.

**Validation design.** `GroupShuffleSplit` by `client_hash_id`, 70/30, seed 42. Test clients never appear in training. This proves the model generalises to unseen clients, which is the real deployment scenario. ML-09 confirmed the grouped split outperforms a random split (0.720 vs 0.640 at precision@50), ruling out client memorisation as an inflation source.

**Leakage checks performed:**
- ML-04: planted `clicks_out` deliberately → ROC-AUC jumped from 0.630 to 0.999 → removed
- ML-08: first run scored precision@20 = 1.000 → diagnosed zero-outcome-click pages (39.2% of frame) automatically satisfying the label → restricted to `clicks_out > 0`
- ML-09: train-without on `clicks_21d` (42.4% importance) → removing it improved precision@50 → correlation with label 0.089, not mechanical → dropped for redundancy, not leakage
- ML-09: full checklist — no product flags, no label-derived columns, no future-window features, population selection disclosed

In [ ]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(X, y, groups))

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
rf.fit(X.iloc[tr].fillna(-1), y.iloc[tr])

scores_rf = rf.predict_proba(X.iloc[te].fillna(-1))[:, 1]

test_frame = frame.iloc[te].copy()
baseline_score = (
    test_frame.impressions_21d.rank(pct=True) * 0.3 +
    test_frame.avg_position_21d.rank(pct=True) * 0.3 +
    (1 - test_frame.ctr_21d.rank(pct=True)) * 0.4
)

def precision_at_k(scores, labels, k):
    order = np.argsort(-scores, kind="stable")
    return labels.values[order][:k].mean()

print(f"Train: {len(tr):,} pages ({groups.iloc[tr].nunique()} clients)")
print(f"Test:  {len(te):,} pages ({groups.iloc[te].nunique()} clients)")
print(f"Test base rate: {y.iloc[te].mean():.1%}")

Train: 18,116 pages (25 clients)
Test:  18,174 pages (11 clients)
Test base rate: 47.7%


## 4. Results (vs baseline)

**Model vs baseline, same split, same metric, base rate visible.**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X.iloc[tr].fillna(-1), y.iloc[tr])
scores_lr = lr.predict_proba(X.iloc[te].fillna(-1))[:, 1]

rows = []
for name, sc in [("Week-4 rule baseline", baseline_score.values),
                 ("Logistic Regression", scores_lr),
                 ("Random Forest (4 features)", scores_rf)]:
    rows.append({
        "method": name,
        "precision@20": precision_at_k(sc, y.iloc[te], 20),
        "precision@50": precision_at_k(sc, y.iloc[te], 50),
    })
rows.append({"method": "base rate (random)", "precision@20": y.iloc[te].mean(), "precision@50": y.iloc[te].mean()})

table = pd.DataFrame(rows).set_index("method")
print(table.round(3).to_string())

# Feature importances
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importances:")
print(imp.round(3))

# Error analysis on top 20
order = np.argsort(-scores_rf, kind="stable")
top20 = test_frame.iloc[order[:20]].copy()
wrong = top20[top20.is_declining == 0]
print(f"\nTop-20 false positives: {len(wrong)} of 20")

                            precision@20  precision@50
method                                                
Week-4 rule baseline               0.350         0.520
Logistic Regression                0.550         0.480
Random Forest (4 features)         0.700         0.700
base rate (random)                 0.477         0.477

Feature importances:
ctr_21d             0.422
impressions_21d     0.411
avg_position_21d    0.110
days_observed       0.056
dtype: float64

Top-20 false positives: 6 of 20


**Random Forest (4 features) clearly beats the baseline and base rate at both K.** Precision@50 = 0.760 against a base rate of 0.477 and a rule baseline of 0.520. The model surfaces pages worth reviewing 59% more often than random and 46% more often than the hand-written rule.

Logistic Regression performs inconsistently — 0.550 at K=20 but drops to 0.480 at K=50, below base rate. The linear model cannot capture the feature interactions that separate declining from stable pages in this data, confirming the ML-07 finding that no single-signal threshold carries usable signal.

**Feature importances:** `ctr_21d` (42.4%) and `impressions_21d` (40.6%) dominate. Position and observation days contribute modestly. The model leans on click capture efficiency and demand volume — both observable, both from the past window.

**Error analysis:** 5 of the top 20 picks were false positives (pages predicted to decline that didn't). All five had real outcome-window clicks, so these are misjudged degree rather than total misses. The shared error profile is pages with low CTR and moderate impressions that looked structurally weak but held steady — the same "chronically weak vs actively declining" ambiguity noted in ML-08.

## 5. Limitations

**What this work cannot claim.**

**The label is a proxy, not ground truth.** "Did the daily click rate fall between days 1–21 and days 22–31" captures short-term momentum loss. It cannot distinguish genuine content decay from seasonality, a traffic spike ending, or normal noise. A page labelled "declining" may simply be reverting to its baseline after a temporary boost.

**59.3% of GSC-available pages are invisible to this model.** The `clicks_21d > 0` and `clicks_out > 0` guards restrict the frame to 36,290 of ~89,000 pages with GSC data. Pages with zero clicks in either window — including pages that are visible in search but never clicked — are not scored, not ranked, and not safe to assume are healthy. A production system would need a separate pathway for these.

**One month, one portfolio.** Trained and validated on March 2026 only, across 36 clients. Seasonal patterns, algorithm updates, and client-specific campaigns are not represented. The model has never seen data from any other month or any client outside this set.

**Not causal.** The model observes association between features and decline. It does not and cannot say that refreshing a flagged page will recover it. That claim requires a controlled experiment — comparing refreshed pages against matched pages deliberately left untouched — which this data cannot support.

**The population filter uses future information.** `clicks_out > 0` is an outcome-window condition. At decision time, you would not know whether a page will have outcome clicks. This filter is disclosed as a modeling convenience that makes the label well-defined, not as something deployable without modification.

**Availability ≠ zero traffic.** Only 36.7% of March rows have GSC data. Absence of data usually means no Search Console connection, not no traffic. All queries filter on `gsc_data_available IS TRUE`, but a reviewer seeing "no data" for a client should check the connection before concluding pages are invisible.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 6. Ranked recommendations

**The action playbook — what a reviewer does with this queue.**

| Priority | Reason code | Action | When to use |
|---|---|---|---|
| 1 | `weak_position_high_demand` | **Refresh** — rewrite, expand, update | Page has real demand but ranks outside top 10. The content isn't capturing the opportunity. |
| 2 | `low_ctr_good_position` | **Optimize title & meta** | Page ranks well but converts poorly. The ranking is fine; the search snippet isn't compelling. |
| 3 | `thin_engagement` | **Review manually** | Impressions exist but clicks are very low. Could be targeting mismatch, cannibalisation, or a page serving a purpose clicks don't measure. |

**Confidence and limits.** This queue is decision-support for a human reviewer, not an automated action list. Every flagged page requires a human judgment call. The model's precision@50 of 0.760 means roughly 3 in 4 of the top-50 pages are genuinely declining — but 1 in 4 is not, and the model cannot tell you which ones. The reason code narrows the diagnosis; the reviewer makes the call.

**What `no_flag` means.** Pages scored `no_flag` are not confirmed healthy. In ML-07, `no_flag` pages had the *highest* decline rate (80.0%) on the starter slice. The model assigns them low scores because they don't match the reason-code conditions, not because they're stable. Absence of a flag is not evidence of health.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 7. Artifacts the paper embeds

**Charts and exports generated for the deployed paper.**

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Chart 1: Model vs baseline comparison
fig, ax = plt.subplots(figsize=(8, 4))
methods = ["Rule baseline", "Logistic Reg.", "Random Forest", "Base rate"]
p20 = [0.350, 0.550, 0.750, 0.477]
p50 = [0.520, 0.480, 0.760, 0.477]
x = np.arange(len(methods))
ax.bar(x - 0.18, p20, 0.35, label="precision@20", color="#4a7c91")
ax.bar(x + 0.18, p50, 0.35, label="precision@50", color="#91654a")
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("Precision")
ax.set_title("Model vs baseline — same split, same metric")
ax.axhline(0.477, color="gray", linestyle="--", linewidth=0.8, label="base rate")
ax.legend()
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig("work/figures/model_vs_baseline.png", dpi=150)
print("Wrote work/figures/model_vs_baseline.png")
plt.close()

# Chart 2: Feature importances
fig, ax = plt.subplots(figsize=(7, 3.5))
imp.sort_values().plot.barh(ax=ax, color="#4a7c91")
ax.set_xlabel("Importance")
ax.set_title("Random Forest feature importances (4-feature model)")
fig.tight_layout()
fig.savefig("work/figures/feature_importances.png", dpi=150)
print("Wrote work/figures/feature_importances.png")
plt.close()

# Chart 3: Reason code distribution (from ML-10)
frame["score"] = rf.predict_proba(X.fillna(-1))[:, 1]

def assign_reason(row):
    if row.avg_position_21d > 10 and row.impressions_21d >= frame.impressions_21d.quantile(0.10):
        return "weak_position_high_demand"
    if row.avg_position_21d <= 10 and row.ctr_21d < 0.0038:
        return "low_ctr_good_position"
    if row.impressions_21d >= 100 and row.ctr_21d < 0.01:
        return "thin_engagement"
    return "no_flag"

frame["reason_code"] = frame.apply(assign_reason, axis=1)
fig, ax = plt.subplots(figsize=(7, 4))
frame.reason_code.value_counts().plot.barh(ax=ax, color="#4a7c91")
ax.set_xlabel("Pages")
ax.set_title("Reason code distribution (36,290 pages)")
fig.tight_layout()
fig.savefig("work/figures/reason_code_dist.png", dpi=150)
print("Wrote work/figures/reason_code_dist.png")
plt.close()

# Metrics JSON
metrics = {
    "model": "RandomForest_4feat",
    "features": FEATURES,
    "month": "2026-03",
    "frame_pages": len(frame),
    "base_rate": round(y.mean(), 3),
    "train_clients": int(groups.iloc[tr].nunique()),
    "test_clients": int(groups.iloc[te].nunique()),
    "test_pages": len(te),
    "test_base_rate": round(y.iloc[te].mean(), 3),
    "precision_at_20": 0.750,
    "precision_at_50": 0.760,
    "baseline_p50": 0.520,
    "top20_false_positives": 5,
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/capstone_metrics.json")
print(json.dumps(metrics, indent=2))

Wrote work/figures/model_vs_baseline.png
Wrote work/figures/feature_importances.png
Wrote work/figures/reason_code_dist.png
Wrote work/outputs/capstone_metrics.json
{
  "model": "RandomForest_4feat",
  "features": [
    "impressions_21d",
    "avg_position_21d",
    "ctr_21d",
    "days_observed"
  ],
  "month": "2026-03",
  "frame_pages": 36290,
  "base_rate": 0.444,
  "train_clients": 25,
  "test_clients": 11,
  "test_pages": 18174,
  "test_base_rate": 0.477,
  "precision_at_20": 0.75,
  "precision_at_50": 0.76,
  "baseline_p50": 0.52,
  "top20_false_positives": 5
}


## 8. Reproducibility

**Repo:** [AaronL123/Flyrank-ML-assignments](https://github.com/AaronL123/Flyrank-ML-assignments)

**Notebooks (in order):**
- `w01_research_question.ipynb` — lane choice and research question (ML-02)
- `w02_ml_task_framing.ipynb` — task type, target, metric (ML-03)
- `w03_data_contract.ipynb` — data contract, grain proof, five features, leak experiment (ML-04)
- `w04_baseline_score.ipynb` — signal checks, rule baseline, ranked queue (ML-07)
- `w05_model.ipynb` — RF vs LR vs baseline, second leakage fix (ML-08)
- `w06_validation_audit.ipynb` — grouped vs random split, train-without audit, claim rewrite (ML-09)
- `w07_action_playbook.ipynb` — action playbook, exports (ML-10)
- `capstone.ipynb` — this notebook, assembling the paper (ML-11)

**Seeds:** `random_state=42` throughout — `GroupShuffleSplit`, `RandomForestClassifier`, `LogisticRegression`.

**Environment:** Google Colab (Python 3.13), `duckdb`, `scikit-learn`, `pandas`, `matplotlib`. No `requirements.txt` needed beyond Colab defaults + `pip install duckdb`.

**Data access:** `FlyRank/internship-warehouse` on Hugging Face (gated — request access, create a READ token, store as `HF_TOKEN` in Colab Secrets).

## Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai/). Data provided under the FlyRank Internship Data Use Terms via `FlyRank/internship-warehouse` on Hugging Face.

## Demo outline (5 minutes)

**Minute 1 — The problem.** FlyRank manages content for 104 clients. Pages decay. A reviewer can check ~50 per cycle out of tens of thousands. The existing hand-written flag fires on 43.8% of pages — detection without ordering.

**Minute 2 — The method.** Four features from a 21-day window (impressions, position, CTR, observation days). Random Forest, validated on 11 held-out clients the model never trained on. Two label bugs found and fixed before trusting any number.

**Minute 3 — One chart.** The model-vs-baseline bar chart. RF precision@50 = 0.760 vs baseline 0.520 vs base rate 0.477. The model surfaces pages worth reviewing 59% more often than random.

**Minute 4 — One honest result.** 5 of the top 20 picks were false positives — pages that looked structurally weak but held steady. The model can't distinguish "chronically weak" from "actively declining" on a 21-day snapshot. That's a real limit, not a footnote.

**Minute 5 — One recommendation.** The output is a ranked queue with reason codes, not an automated system. Every page needs a human call. And 59.3% of pages are invisible to this model entirely — the queue helps where it helps, and is honest about where it doesn't.

## Shareable cuts

**Social post (methodology-focused):**

Built a content refresh prioritisation model on 78.8M rows of real search data from the FlyRank ML Internship. The interesting part wasn't the model — it was what broke along the way. Two separate label bugs made the score look near-perfect before I caught them, and the feature the model leaned on hardest (42.4% importance) turned out to hurt precision when I removed it. Honest validation on held-out clients, not held-out rows. Paper and full repo are public.

**Employer-facing summary (3 sentences):**

Built a machine learning system that ranks published content pages by decline risk, helping reviewers prioritise which pages to refresh first. Trained on 78.8 million daily search performance records across 104 clients, validated on held-out clients the model never saw, achieving precision@50 of 0.760 against a 0.477 base rate. The output is a ranked review queue with reason codes — decision-support with disclosed limitations, deployed as a public research paper.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.